# Notebook 02 — Single-Sample SGD with MAE

**Session:** Linear Regression — L1, L2, and Huber Losses (Level 2)  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement SGD with **Mean Absolute Error (MAE)**.  The key difference from MSE is that the
gradient is the *sign* of the residual rather than the residual itself.

**Learning objectives:**
- Understand why MAE is not differentiable at zero residual.
- Implement the sign-based gradient correctly.
- Compare convergence behaviour against the MSE notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 1  Load Data

In [ ]:
def get_column(data: list, index: int) -> list:
    return [row[index] for row in data]


data = np.genfromtxt("../data/data.csv", delimiter=",").tolist()
x_data = get_column(data, 0)
y_data = get_column(data, 1)
N = len(x_data)

print(f"areas  : {x_data}")
print(f"prices : {y_data}")
print(f"N      : {N}")

## 2  Model and Loss Functions

In [ ]:
def predict(x: float, w: float, b: float) -> float:
    return w * x + b


def gradient_mae(y_hat: float, y: float, x: float) -> tuple[float, float]:
    """Per-sample MAE sub-gradient.

    dL/dw = x * sign(y_hat - y)  =  x * (y_hat - y) / |y_hat - y|
    dL/db =     sign(y_hat - y)  =      (y_hat - y) / |y_hat - y|

    Note: undefined when y_hat == y; in practice this occurs very rarely
    during floating-point training.
    """
    residual = y_hat - y
    dw = x * residual / abs(residual)
    db = residual / abs(residual)
    return dw, db


def update_weight(w: float, b: float, lr: float, dw: float, db: float) -> tuple[float, float]:
    return w - lr * dw, b - lr * db

## 3  SGD Training Loop

In [ ]:
w: float = -0.34
b: float = 0.04
lr: float = 0.01
epoch_max: int = 20

losses: list[float] = []

for _epoch in range(epoch_max):
    for i in range(N):
        x = x_data[i]
        y = y_data[i]

        y_hat = predict(x, w, b)
        loss = abs(y_hat - y)
        losses.append(loss)

        dw, db = gradient_mae(y_hat, y, x)
        w, b = update_weight(w, b, lr, dw, db)

print(f"Final parameters  →  w = {w:.4f}, b = {b:.4f}")

## 4  Loss Curve and Fitted Line

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("MAE loss")
plt.title("Per-sample MAE loss (SGD)")
plt.tight_layout()
plt.show()

In [ ]:
x_line = list(range(2, 8))
y_line = [predict(x, w, b) for x in x_line]

plt.figure(figsize=(6, 4))
plt.scatter(x_data, y_data, label="Observations")
plt.plot(x_line, y_line, "r-", label=f"Fit: y = {w:.3f}x + {b:.3f}")
plt.xlabel("Area (×100 m²)")
plt.ylabel("Price (Tael)")
plt.title("Linear Regression — MAE (SGD)")
plt.xlim(3, 7)
plt.ylim(4, 10)
plt.legend()
plt.tight_layout()
plt.show()